# Tool Calling

In [1]:
from agents import Agent, ModelSettings, Runner, function_tool, trace
from setup import bedrock_tool

Create a static calorie table that we can use as a tool:

In [2]:
@function_tool
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [3]:
# get_food_calories('banana')

In [4]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(get_food_calories.__dict__)],
)

In [5]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)



A banana and an apple together have 185 calories.


Enforce tools use:

In [6]:
# Note: Bedrock does not support tool_choice enforcement — the agent will still call the tool naturally.
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    Always use the get_food_calories tool to look up calorie information.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(get_food_calories.__dict__)],
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)



A banana has 105 calories and an apple has 80 calories, for a total of 185 calories.
